# Práctica RAG - Fase 1: Limpieza y Chunking de Documentos

> **Proyecto:** Benchmark de Embeddings y LLMs Locales para RAG  
> **Autores (Alumnos):** Alfred, Ane y Naiara  
> **Hardware de referencia:** GPU NVIDIA A800 (Entorno local con PyTorch y Ollama)  

---

### ¿Qué hacemos en este notebook?
En esta primera fase nos encargamos de preparar los dos corpus del proyecto para que los modelos de embeddings puedan indexarlos y buscar en ellos de manera eficiente:
1. **Normativa jurídica:** *Estatuto de los Trabajadores* (BOE consolidado, RDL 2/2015).
2. **Documentación técnica:** *Manual del Administrador de Debian* (edición en español para Debian 11).

### ¿Por qué usamos EPUB en lugar de PDF directo?
Los dos documentos oficiales los descargamos en formato **EPUB**. Un archivo  es en realidad un archivo comprimido  que contiene archivos XHTML/HTML bien estructurados. Al procesar el HTML directamente:
- **Evitamos el ruido del OCR y de cabeceras/pies de página** que suelen ensuciar los PDF.
- **Mantenemos los títulos y números de artículo intactos**, lo que es fundamental para que luego el LLM pueda citar la fuente con precisión.
- Lo resolvemos con herramientas estándar de Python (, , ), sin dependencias pesadas.

### Nuestra estrategia de Chunking (Troceado inteligente)
No queríamos trocear el texto de manera ciega cortando oraciones a la mitad, así que aplicamos las siguientes reglas:
- **Respetar la estructura natural:** cada artículo del BOE o sección de Debian se toma como unidad base.
- **Límite de 1.400 caracteres:** si el artículo o sección cabe en este tamaño, lo dejamos intacto en un solo fragmento.
- **Partición por párrafos con solapamiento (*overlap*):** si el texto es más largo, lo dividimos en fragmentos de **1.200 caracteres con 200 caracteres de solapamiento** para garantizar que no se pierda el contexto entre fragmentos adyacentes.
- **Inyección de título:** a cada fragmento le añadimos al principio el encabezado  o  para que el modelo de embedding entienda el tema aunque el párrafo sea muy específico.

In [1]:
# Librerías que usamos para abrir los EPUB, limpiar el texto y guardar los resultados.
import zipfile
import re
import html
import json
import os
from html.parser import HTMLParser

# Elegimos las carpetas según desde dónde se ejecute el notebook: desde data/ o desde la raíz del proyecto.
RAW_DIR = "raw" if os.path.exists("raw") else "data/raw"
OUT_DIR = "processed" if os.path.exists("raw") else "data/processed"
# Crea la carpeta de salida si todavía no existe.
os.makedirs(OUT_DIR, exist_ok=True)

# Esta clase recorre el HTML del EPUB y va guardando solo el texto que nos interesa.
class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        # Aquí se guardan trozos de texto según se van encontrando.
        self.pieces = []
        # Estas etiquetas suelen separar bloques (párrafos, títulos, listas...).
        self.block_tags = {"p", "div", "h1", "h2", "h3", "h4", "h5", "h6", "li", "br", "tr"}

    # HTMLParser llama a este método cuando encuentra una etiqueta de apertura, como <p>.
    def handle_starttag(self, tag, attrs):
        if tag in self.block_tags:
            self.pieces.append("\n")

    # Se añade también un salto al cerrar el bloque para separar el texto.
    def handle_endtag(self, tag):
        if tag in self.block_tags:
            self.pieces.append("\n")

    # Este método recibe el texto que hay entre las etiquetas HTML.
    def handle_data(self, data):
        self.pieces.append(data)

    # Junta los trozos y quita espacios y saltos de línea repetidos.
    def get_text(self):
        text = "".join(self.pieces)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n+", "\n\n", text)
        return text.strip()

# Divide un texto largo por párrafos. El solapamiento intenta repetir algo de texto
# entre fragmentos para que no se pierda el contexto al separarlos.
def chunk_text(text, max_chars=1400, overlap=200):
    # Los párrafos están separados por dos saltos de línea.
    paragraphs = text.split("\n\n")
    chunks = []
    current_chunk = []
    current_len = 0

    # Vamos añadiendo párrafos hasta acercarnos al máximo permitido.
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        # Si el párrafo ya no cabe, cerramos el fragmento actual y empezamos otro.
        if current_len + len(p) > max_chars and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            # Si el último párrafo es corto, lo repetimos al principio del siguiente
            # fragmento para conservar algo de contexto.
            if len(current_chunk[-1]) <= overlap:
                current_chunk = [current_chunk[-1], p]
                current_len = len(current_chunk[0]) + len(p)
            else:
                current_chunk = [p]
                current_len = len(p)
        else:
            current_chunk.append(p)
            current_len += len(p)

    # No hay que olvidar guardar el último fragmento.
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))
    return chunks

## 1. Procesamiento del Documento 1: Normativa (BOE Estatuto de los Trabajadores)

In [2]:
# Procesa el EPUB del BOE y devuelve una lista de fragmentos listos para usar.
def process_boe():
    # Un EPUB es un ZIP: abrimos el archivo y leemos dentro el XHTML del BOE.
    with zipfile.ZipFile(os.path.join(RAW_DIR, "BOE-A-2015-11430-consolidado.epub"), "r") as z:
        data = z.read("OEBPS/conso-2015-11430.xhtml").decode("utf-8")

    # Buscamos los párrafos del documento y su clase, que indica si son títulos o texto.
    matches = re.findall(r"<p\b[^>]*class=\"([^\"]*)\"[^>]*>(.*?)</p>", data, re.DOTALL)

    # Vamos agrupando el contenido por artículo o anexo.
    sections = []
    current_ref = "Preámbulo"
    current_title = "Preámbulo"
    current_body = []

    # Recorremos los párrafos encontrados en el mismo orden que aparecen.
    for cls, content in matches:
        # Quitamos etiquetas HTML y convertimos caracteres especiales, como &amp;, a texto normal.
        clean = html.unescape(re.sub(r"<[^>]+>", " ", content)).strip()
        clean = re.sub(r"\s+", " ", clean)
        if not clean:
            continue

        # Cuando aparece un artículo o anexo nuevo, guardamos el anterior y empezamos otro.
        if "articulo" in cls or "anexo" in cls:
            if current_body:
                sections.append((current_ref, current_title, "\n\n".join(current_body)))
                current_body = []
            current_title = clean
            m_ref = re.search(r"(Art[ií]culo\s+\d+[\w\.]*|Disposici[oó]n\s+[\w\s\.]+|Anexo\s+[\w\s\.]*)", clean, re.I)
            current_ref = m_ref.group(1).strip() if m_ref else clean[:30]
        else:
            current_body.append(clean)

    if current_body:
        sections.append((current_ref, current_title, "\n\n".join(current_body)))

    # Convertimos las secciones en fragmentos y les ponemos identificadores únicos.
    chunks = []
    chunk_idx = 0
    for ref, title, body in sections:
        # Las secciones cortas se dejan enteras; las largas se parten por párrafos.
        if len(body) <= 1400:
            full_content = f"[{title}]\n{body}".strip()
            chunks.append({
                "id": f"boe_{chunk_idx:04d}",
                "doc": "BOE-Estatuto-Trabajadores",
                "reference": ref,
                "title": title,
                "text": full_content
            })
            chunk_idx += 1
        else:
            sub_chunks = chunk_text(body, max_chars=1200, overlap=200)
            for i, sc in enumerate(sub_chunks):
                full_content = f"[{title} - Parte {i+1}/{len(sub_chunks)}]\n{sc}".strip()
                chunks.append({
                    "id": f"boe_{chunk_idx:04d}",
                    "doc": "BOE-Estatuto-Trabajadores",
                    "reference": ref,
                    "title": f"{title} (Parte {i+1})",
                    "text": full_content
                })
                chunk_idx += 1

    # Guardamos la lista como JSON, un formato de texto que conserva listas y campos.
    with open(os.path.join(OUT_DIR, "boe_chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    return chunks

boe_chunks = process_boe()
print(f"BOE procesado correctamente: {len(boe_chunks)} fragmentos guardados en {OUT_DIR}/boe_chunks.json")

BOE procesado correctamente: 420 fragmentos guardados en processed/boe_chunks.json


## 2. Procesamiento del Documento 2: Técnica (Manual del Administrador de Debian)

In [3]:
# Procesa los capítulos HTML del manual de Debian y devuelve sus fragmentos.
def process_debian():
    chunks = []
    chunk_idx = 0

    # Abrimos el EPUB y buscamos sus capítulos HTML en orden alfabético.
    with zipfile.ZipFile(os.path.join(RAW_DIR, "debian-handbook.epub"), "r") as z:
        files = [f for f in sorted(z.namelist()) if f.startswith("OEBPS/ch") and f.endswith(".html")]
        for f in files:
            # Leemos el capítulo actual como texto.
            raw = z.read(f).decode("utf-8")
            # El título y la referencia ayudan a identificar cada fragmento después.
            t_m = re.search(r"<title>(.*?)</title>", raw, re.I)
            raw_title = html.unescape(t_m.group(1)).strip() if t_m else f

            ref_m = re.search(r"(\d+\.\d+(?:\.\d+)?|Capítulo\s+\d+)", raw_title)
            ref = ref_m.group(1) if ref_m else raw_title[:20]

            # Usamos el extractor definido antes para pasar de HTML a texto limpio.
            parser = TextExtractor()
            parser.feed(raw)
            body = parser.get_text()

            # A veces el título también aparece al principio del cuerpo; lo quitamos
            # para no repetirlo (ya lo guardamos aparte).
            if body.startswith(raw_title):
                body = body[len(raw_title):].strip()

            # Los capítulos cortos se guardan enteros y los largos se dividen.
            if len(body) <= 1400:
                full_content = f"[{raw_title}]\n{body}".strip()
                chunks.append({
                    "id": f"deb_{chunk_idx:04d}",
                    "doc": "Debian-Handbook",
                    "reference": ref,
                    "title": raw_title,
                    "text": full_content
                })
                chunk_idx += 1
            else:
                sub_chunks = chunk_text(body, max_chars=1200, overlap=200)
                for i, sc in enumerate(sub_chunks):
                    full_content = f"[{raw_title} - Parte {i+1}/{len(sub_chunks)}]\n{sc}".strip()
                    chunks.append({
                        "id": f"deb_{chunk_idx:04d}",
                        "doc": "Debian-Handbook",
                        "reference": ref,
                        "title": f"{raw_title} (Parte {i+1})",
                        "text": full_content
                    })
                    chunk_idx += 1

    # Guardamos todos los fragmentos del manual en un archivo JSON.
    with open(os.path.join(OUT_DIR, "debian_chunks.json"), "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)
    return chunks

debian_chunks = process_debian()
print(f"Debian procesado correctamente: {len(debian_chunks)} fragmentos guardados en {OUT_DIR}/debian_chunks.json")

Debian procesado correctamente: 1210 fragmentos guardados en processed/debian_chunks.json


## 3. Resumen de los Documentos Procesados

In [4]:
# Pandas se usa únicamente aquí para mostrar una tabla resumen.
import pandas as pd
from IPython.display import display

# Cada diccionario representa una fila de la tabla, con datos de un documento.
resumen = pd.DataFrame([
    {
        "Documento": "1. Normativa (BOE Estatuto de los Trabajadores)",
        "Archivo Original": "BOE-A-2015-11430-consolidado.epub",
        "Fragmentos": len(boe_chunks),
        "Longitud Media (caracteres)": round(sum(len(c["text"]) for c in boe_chunks) / len(boe_chunks), 1)
    },
    {
        "Documento": "2. Técnica (Debian Handbook)",
        "Archivo Original": "debian-handbook.epub",
        "Fragmentos": len(debian_chunks),
        "Longitud Media (caracteres)": round(sum(len(c["text"]) for c in debian_chunks) / len(debian_chunks), 1)
    }
])
# Mostramos la tabla sin índice y con una cifra decimal en la longitud media.
display(resumen.style.format({"Longitud Media (caracteres)": "{:.1f}"}).hide(axis="index"))

Documento,Archivo Original,Fragmentos,Longitud Media (caracteres)
1. Normativa (BOE Estatuto de los Trabajadores),BOE-A-2015-11430-consolidado.epub,420,939.1
2. Técnica (Debian Handbook),debian-handbook.epub,1210,1029.0
